# SNS Studio: one real GPU job
指定したJob/attemptだけ実行します。全Job巡回、keep-alive、追加購入、有料API、SNS投稿は行いません。未入力なら推論しません。
Google認証・同意が必要な操作は利用者自身が行ってください。実GPU検証済みという意味ではなく、実行結果を検証するNotebookです。

In [ ]:
EXECUTION_MODE = "real"
TARGET_JOB_ID = ""
TARGET_ATTEMPT = 1
MANIFEST_FILE_ID = ""
INFERENCE_TIMEOUT_SECONDS = 1800
assert EXECUTION_MODE == "real"


## GPU / RAM / disk事前検査
Colabのランタイム設定はPython 3.12（例: 2026.01）/ GPU。既存契約内の構成だけを使います。モデルごとの必要資源はJob確認時にも再検査します。

In [ ]:
import json, sys, shutil, os
import torch, psutil
assert sys.version_info[:2] in ((3,11),(3,12)), "Use the pinned Python runtime"
assert torch.cuda.is_available(), "BLOCKED_BY_GPU"
free,total=torch.cuda.mem_get_info()
print(json.dumps(dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),vram_total_GiB=total/2**30,vram_free_GiB=free/2**30,ram_free_GiB=psutil.virtual_memory().available/2**30,disk_free_GiB=shutil.disk_usage("/content").free/2**30),indent=2))
assert not any(os.getenv(k, "").lower() not in ("", "0", "false") for k in ("MOCK_GENERATION","SNS_STUDIO_ALLOW_TEST_MOCK")), "Mock override forbidden"


## 固定依存をColabの専用venvへ導入
Windows/4017の環境は変更しません。既存Colab kernelの依存は変更せず、推論子プロセスだけが専用venvを使います。モデルweightsはこのセルでは取得しません。

In [ ]:
import subprocess, sys
from pathlib import Path
VENV=Path("/content/sns-studio-inference-venv")
subprocess.run([sys.executable,"-m","venv","--without-pip",str(VENV)],check=True,timeout=120)
INFERENCE_PYTHON=str(VENV/"bin/python")
subprocess.run([sys.executable,"-m","pip","--python",INFERENCE_PYTHON,"install","--disable-pip-version-check","--no-input","pip==25.1.1"],check=True,timeout=120)
PINNED_REQUIREMENTS = ['torch==2.6.0', 'torchvision==0.21.0', 'diffusers==0.35.1', 'transformers==4.51.3', 'accelerate==1.6.0', 'huggingface-hub==0.34.4', 'safetensors==0.5.3', 'sentencepiece==0.2.0', 'protobuf==5.29.4', 'Pillow==11.2.1', 'imageio==2.37.0', 'imageio-ffmpeg==0.6.0', 'ftfy==6.3.1', 'psutil==7.0.0', 'google-api-python-client==2.170.0', 'google-auth==2.40.1']
subprocess.run([INFERENCE_PYTHON,"-m","pip","install","--disable-pip-version-check","--no-input",*PINNED_REQUIREMENTS],check=True,timeout=900)
subprocess.run([INFERENCE_PYTHON,"-m","pip","check"],check=True,timeout=60)


## 最小実行bundleをアップロード・hash照合
ローカルで `python services/colab/build_bundle.py --output sns-studio-runner-bundle.zip` を実行して作ったbundleだけを選びます。private repository全体や.envはアップロードしません。

In [ ]:
from google.colab import files
import hashlib, io, zipfile, json
from pathlib import Path
BUNDLE_SHA256 = '3385e3da01cf906fb367b22e88e2885eb9a0f9b6b31a45315f6f965f186727b5'
uploaded=files.upload()
assert len(uploaded)==1
data=next(iter(uploaded.values()))
assert hashlib.sha256(data).hexdigest()==BUNDLE_SHA256,"Bundle source mismatch"
RUNNER=Path("/content/sns-studio-runner")
RUNNER.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(data)) as z:
    manifest=json.loads(z.read("bundle-manifest.json"))
    expected={"sns_studio_model_adapter.py","model-contract.json","requirements-colab.txt","drive_one_shot.py","run_inference.py"}
    assert set(manifest)==expected and set(z.namelist())==expected|{"bundle-manifest.json"}
    for name in sorted(expected):
        content=z.read(name)
        assert hashlib.sha256(content).hexdigest()==manifest[name]
        (RUNNER/name).write_bytes(content)
import sys
sys.path.insert(0,str(RUNNER))
subprocess.run([INFERENCE_PYTHON,"-c",f"import sys;sys.path.insert(0,{str(RUNNER)!r});from sns_studio_model_adapter import verify_runtime_versions;verify_runtime_versions()"],check=True,timeout=60)
print("Runner source hashes and pinned dependencies verified")


## Google Drive接続（利用者の操作）
既存のSNS Studio用Driveフォルダ・指定Jobだけを使います。ログインや同意画面は利用者自身で完了してください。tokenはNotebookセルやmanifestに保存しません。

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth, httplib2
from google_auth_httplib2 import AuthorizedHttp
from googleapiclient.discovery import build
credentials,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive=build("drive","v3",http=AuthorizedHttp(credentials,http=httplib2.Http(timeout=60)),cache_discovery=False)


## 指定Job / 入力素材・採用設定確認
UIから得たJob ID、attempt、manifest file IDを最初のセルへ設定します。ローカル/uploadsや任意URLはColab入力にしません。

In [ ]:
import re
assert re.fullmatch(r"[A-Za-z0-9_-]{1,100}",TARGET_JOB_ID),"Set target Job ID"
assert re.fullmatch(r"[A-Za-z0-9_-]{1,200}",MANIFEST_FILE_ID),"Set manifest file ID"
from drive_one_shot import DriveTransport
from sns_studio_model_adapter import validate_input, canonical_hash, check_gpu_environment
transport=DriveTransport(drive)
job=transport.json(MANIFEST_FILE_ID)
assert job["jobId"]==TARGET_JOB_ID and job["attempt"]==TARGET_ATTEMPT
assert canonical_hash(job["input"])==job["inputHash"]
settings=validate_input(job["input"])
assert settings["mock"] is False
print(json.dumps(dict(jobId=TARGET_JOB_ID,attempt=TARGET_ATTEMPT,mode=settings["mode"],model=settings["modelName"],adopted=settings["adopted"],seed=settings["seed"],resources=check_gpu_environment()),indent=2))


## 指定Jobを1件だけ実行
Drive事前予約IDのcreate競合で二重実行を防ぎます。取消は結果取り込みを拒否し、推論の即時停止は保証しません。推論途中からの再開は非対応。完成出力が同じランタイムに残っている場合だけ再upload/同期できます。失敗推論の再実行はSNS Studioで新attemptを作ります。

In [ ]:
from drive_one_shot import run_one_job
result=run_one_job(transport,MANIFEST_FILE_ID,TARGET_JOB_ID,TARGET_ATTEMPT,Path("/content/sns-studio-jobs"),timeout_seconds=INFERENCE_TIMEOUT_SECONDS,python_executable=INFERENCE_PYTHON)
print(json.dumps({k:result[k] for k in ("jobId","attempt","status","inputHash","mock")},indent=2))


## 出力検証 / Drive保存 / 結果manifest
run_one_jobはffprobe・全フレームdecode・SHAを確認した実MP4だけをprivate Driveへ保存します。同じIDによる再uploadは重複作成しません。実行モデル・revision・mode・画像hash・使用量・実測尺はexecutionへ記録します。

In [ ]:
from IPython.display import Video,display
work=Path("/content/sns-studio-jobs")/TARGET_JOB_ID/f"attempt-{TARGET_ATTEMPT}"
summary=json.loads((work/"generation-evidence.json").read_text())
print(json.dumps(summary,indent=2))
output=next(work.glob("sns_studio_*.mp4"))
display(Video(str(output),embed=True))


## 人間による確認とSNS Studio同期
最後まで再生し、破損・黒画面・静止画ループ・品質を確認してください。無音モデルに架空の音声を足しません。SNS Studioで同期→Preview→制作Queue承認→共通投稿画面への引継ぎを確認し、投稿・予約は確定しません。Notebook成功と管理画面E2E成功は別判定です。